# Topic 1 — Prompt Chaining (Micro-Intent → First-Hand Experience → EEAT/YMYL Audit)

Self-contained Colab port of `topic-1-prompt-chaining/` from the
`project-2026-01-claude` repo, as of 2026-09-12.

**Architecture** (unchanged from the source project):
- **Deterministic control flow** — Python, ported verbatim from `src/orchestrator.py`
  and `src/schemas.py`. This is what runs the same way every time: validates each
  stage's output against a pydantic schema, drives the Stage-C audit → correct →
  re-audit retry loop, and persists session state to disk.
- **Dynamic components** — the actual LLM calls (Stage A intent modeling, Stage B
  first-hand-experience generation, Stage C audit + correction), backed here by
  **Azure OpenAI** via a port of `src/llm_client.py`'s `AzureOpenAILLMClient`.

**What's ported vs. adapted for Colab:**
| Source file | Ported as | Adapted? |
|---|---|---|
| `src/schemas.py` | Cell 2 | No — verbatim |
| `prompts/*.md` + `src/prompts.py` | Cell 3 | No — verbatim |
| `src/llm_client.py` (`AzureOpenAILLMClient` only) | Cell 4 | Credentials come from **Colab Secrets** (`google.colab.userdata`) instead of `.env` |
| `src/orchestrator.py` | Cell 5 | `SESSIONS_DIR` points at `./sessions` in the Colab runtime instead of a path relative to `__file__` (which doesn't exist the same way in a notebook) |

**Before running:** open the key icon (🔑 Secrets) in Colab's left sidebar and
add these four secrets, then toggle "Notebook access" on for each:
- `AZURE_OPENAI_API_KEY`
- `AZURE_OPENAI_ENDPOINT`
- `AZURE_OPENAI_API_VERSION`
- `AZURE_OPENAI_DEPLOYMENT`

(`AZURE_OPENAI_EMBEDDING_DEPLOYMENT`, if you have it set up, isn't used here —
this chain only does text completion, no embeddings/RAG.) Nothing is written
to disk except the session JSON under `./sessions/` (Colab's local, ephemeral
filesystem) — the API key itself is only held in the notebook's runtime
memory for this session, never printed or written to a file.

**Colab is ephemeral** — the `./sessions/*.json` file (this run's full record:
Stage A intents, the selected one, the pre-correction draft, every audit/
correction cycle, and the final paragraph) will be lost when the runtime
recycles. The last cell downloads it to your machine.

In [ ]:
!pip install -q "pydantic>=2.0.0" "openai>=1.40.0"


## 1. Credentials — read from Colab Secrets, never typed or saved to disk

Uses [Colab's Secrets manager](https://colab.research.google.com) (`google.colab.userdata`) instead of `getpass`/`input()`. Add the four secrets listed above via the 🔑 icon in the left sidebar first, and grant this notebook access to each when prompted — then just run this cell.

In [ ]:
import json
import os
import uuid
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, List

from pydantic import BaseModel, Field, field_validator
from google.colab import userdata

# AZURE_OPENAI_EMBEDDING_DEPLOYMENT is intentionally not read here — this
# chain has no embeddings/RAG step, only text completion.
REQUIRED_SECRETS = [
    "AZURE_OPENAI_API_KEY",
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_API_VERSION",
    "AZURE_OPENAI_DEPLOYMENT",
]

for _name in REQUIRED_SECRETS:
    try:
        os.environ[_name] = userdata.get(_name)
    except userdata.SecretNotFoundError:
        raise RuntimeError(
            f"Colab secret '{_name}' isn't set. Add it via the 🔑 Secrets "
            "panel in the left sidebar, then re-run this cell."
        ) from None
    except userdata.NotebookAccessError:
        raise RuntimeError(
            f"Colab secret '{_name}' exists but this notebook hasn't been "
            "granted access yet — toggle notebook access on for it in the "
            "Secrets panel, then re-run this cell."
        ) from None

print("Loaded Azure OpenAI config from Colab secrets:", ", ".join(REQUIRED_SECRETS))


## 2. Schemas — pydantic contracts for each stage's output

Ported verbatim from `topic-1-prompt-chaining/src/schemas.py`.

In [ ]:
class MicroIntent(BaseModel):
    """Schema for one entry in Stage A's output — a single micro-intent.

    Declares the `{intent, evidence, competitive_rationale}` shape
    `prompts.stage_a_prompt()` asks the LLM to return per entry (Output shape
    for B9). All three fields are required strings; missing or wrong-typed
    data raises `pydantic.ValidationError` on construction. No side effects —
    a plain, validated data container.
    """

    intent: str
    evidence: str
    competitive_rationale: str


class StageAOutput(BaseModel):
    """Output of C2 — Stage A: Micro-Intent Modeling. Source: B7 (+ B8, B9).

    Wraps the full list of micro-intents. `intents: List[MicroIntent]` gives
    nested validation for free — pydantic validates every element against
    `MicroIntent`'s schema too, so one malformed intent fails the whole
    `StageAOutput`, not just that entry.
    """

    intents: List[MicroIntent]

    @field_validator("intents")
    @classmethod
    def at_least_four(cls, v: List[MicroIntent]) -> List[MicroIntent]:
        """Enforces B8 (Stage A must cover >=4 distinct micro-intents) — a
        constraint plain type-checking can't express (`List[MicroIntent]`
        alone would accept a list of length 1). Runs automatically right
        after the basic type-check on `intents` passes.

        `cls` (not `self`): pydantic v2 validators are classmethods, since
        validation happens during construction, before an instance fully
        exists.

        Output/side effect: returns `v` unchanged when the length check
        passes (a validator must return the value it wants kept); otherwise
        raises `ValueError`, which pydantic re-raises as part of a
        `ValidationError` on the whole model — this is what propagates up to
        orchestrator.py's `StageAOutput.model_validate(...)` call and stops
        the run with a clear cause rather than silently accepting too few
        intents. No I/O.
        """
        if len(v) < 4:
            raise ValueError(f"B8 requires >=4 micro-intents, got {len(v)}")
        return v


class StageBOutput(BaseModel):
    """Output of C3 — Stage B: First-hand Experience. Source: B10 (+ B11-13).

    Deliberately the simplest schema here: just `paragraph: str`, no custom
    validation. Stage B's real constraints (B11 sensory detail + objective
    critique, B12 no AI disclaimers/hollow praise, B13 veteran-player
    persona) are all qualitative properties of the text's *content* — not
    something a type schema can mechanically check. That's why they're
    enforced instead by Stage C's audit (`AuditVerdict`, below) actually
    reading the text, not by this schema.
    """

    paragraph: str


class AuditVerdict(BaseModel):
    """Output of C4a — Stage C Audit. Source: B15 (check half).

    The structured verdict orchestrator.py's retry loop branches on
    (`if verdict.passed: break`).
    """

    # Accessed in Python as `.passed` (can't be `.pass` — reserved keyword),
    # but read/written over JSON as "pass" via this alias, matching the exact
    # key stage_c_audit_prompt() instructs the LLM to return.
    passed: bool = Field(alias="pass")
    # Specific EEAT/YMYL violation explanations; defaults to an empty list if
    # the LLM omits the key rather than failing validation outright. Expected
    # empty when `passed` is True.
    reasons: List[str] = Field(default_factory=list)
    # Suggested improvements to the Authoritativeness signal specifically
    # (B15's "improve its Authoritativeness signal" wording), same
    # default-empty pattern as `reasons`.
    authoritativeness_suggestions: List[str] = Field(default_factory=list)

    # Lets the model ALSO be constructed with the real field name (passed=)
    # in addition to the alias (pass=) — used by orchestrator.py's
    # `AuditVerdict(**{"pass": False})`, which needs a dict key since
    # `pass=False` as a literal keyword argument would be a SyntaxError.
    model_config = {"populate_by_name": True}


class StageCCorrection(BaseModel):
    """Output of C4b — Stage C Correction. Source: B15 (correct half) + B14.

    Structurally identical to StageBOutput: wraps the single revised
    `paragraph: str`. No custom validation, for the same reason — whether the
    revision actually resolved the audit's flagged `reasons` is a qualitative
    judgment made by the *next* audit pass (C4a again), not something this
    schema checks.
    """

    paragraph: str


## 3. Prompt templates + prompt-rendering functions

The four templates below are embedded **verbatim** from `topic-1-prompt-chaining/prompts/*.md` (Role/Goal/Instructions for each of C2, C3, C4a, C4b), each containing a `{keyword}` placeholder substituted per run — no keyword is hardcoded. The rendering functions are ported from `src/prompts.py` (adapted only in that templates come from the constants above instead of reading files off disk).

In [ ]:
STAGE_A_TEMPLATE = """<!--
Traceability: prompts/master-prompt.md → build-topic-executables, {#}=1, component C2
Source: topic-1-prompt-chaining/docs/problem-statement-categorized.md
  Goal: B7 (+ B8, B9 constraints; B1, B2, B6 reference)
Type: dynamic

Run note: B6 originally read "Stage A's fixed input is the keyword [娛樂城]"
— a single hardcoded keyword, not a parameter. This template has been
generalized to accept any keyword via the {keyword} placeholder (substituted
by src/prompts.py's stage_a_prompt(keyword)), so the chain isn't hardcoded to
one term. This is an engineering decision beyond B6's literal wording, not a
sourced requirement — flagged here the same way other non-sourced decisions
are flagged in component-specs.md's run notes. B1's specific competitiveness
claim about [娛樂城] is kept below as the worked example this design was
originally built against, not asserted as automatically true of every
substituted keyword.
-->

# Stage A — Micro-Intent Modeling

## Role
You are a senior SEO systems engineer: someone who combines deep, practical SEO
knowledge with the ability to design and build the AI systems that act on that
knowledge. You don't just know what makes content rank or what makes content
trustworthy — you understand that in this domain, those two forces are
frequently in tension, and you build systems whose job is to operate inside
that tension rather than pretend it doesn't exist.

(Default persona from the categorized doc's Role section — no Stage-A-specific
persona was given in the source.)

## Goal
Analyze the keyword [{keyword}] and produce at least 4 distinct layers of genuine
Micro Search Intent, each with evidence-based reasoning for why it was chosen
and the SEO-competition logic behind it. (B7)

## Instructions
- **Constraints:**
  - Cover at least 4 distinct layers/levels of intent (B8) — examples given in
    source (withdrawal-speed anxiety, agent trust verification, game-UI
    immersion needs) are illustrative, not exhaustive or required verbatim.
  - For each intent, give evidence-based reasoning for why it was chosen and
    the underlying SEO-competition logic (B9) — a label alone is not enough.
- **Reference:**
  - This design was originally built against [娛樂城], one of the most
    SEO-competitive keywords in its industry (B1) — when analyzing a
    different keyword, apply the same rigor regardless of whether it shares
    that exact competitive intensity.
  - Google increasingly favors content demonstrating Experience over plain
    product/brand listicles or comparisons (B2).
  - This stage's input is the keyword [{keyword}], provided per run
    (generalized from B6's original fixed-keyword wording — see run note
    above).
- **Output shape** (feeds B16 — the Stage A/B/C results deliverable):
  - One entry per micro-intent: `{intent, evidence, competitive_rationale}`.
  - At least 4 entries.
- **Self-check before finalizing** (from B20, B21):
  - Does each intent include a specific detail a generic competitor analysis
    wouldn't produce (B20 — Detail-Oriented)?
  - Does the same competitive logic visibly connect each intent back to the
    keyword's difficulty (B21 — Logical Consistency)?"""

STAGE_B_TEMPLATE = """<!--
Traceability: prompts/master-prompt.md → build-topic-executables, {#}=1, component C3
Source: topic-1-prompt-chaining/docs/problem-statement-categorized.md
  Goal: B10 (+ B11, B12, B13 constraints)
Type: dynamic

Run note: B13 requires this stage's OUTPUT to read as a veteran Taiwanese
player's independent account. The categorized doc's Role section instead now
holds a general "senior SEO systems engineer" persona (a builder/architect
framing, not a content voice). Using that persona here would actively work
against B12/B13's requirement to suppress AI/analyst tells and sound like a
genuine hobbyist, so this component's Role is built directly from B13,
overriding the doc's default Role for this component only. Flagged here
rather than silently decided, since it's the one place a sourced requirement
(B13) and the doc's current Role content point in different directions.

Also genericized: the Role/self-check below previously hardcoded "娛樂城" as
the platform type. Replaced with a {keyword} placeholder (substituted by
src/prompts.py's stage_b_prompt) so this persona generalizes to whatever
keyword Stage A ran against, while keeping B13's Taiwan-resident/veteran-
player framing intact — that framing is a sourced requirement, the specific
platform noun was not.
-->

# Stage B — First-hand Experience Generation

## Role
You are a resident of Taiwan, an experienced (資深) {keyword} player with real
hands-on history across multiple platforms — not an SEO analyst, marketer, or AI assistant. You write the way a genuine hobbyist shares an account with peers: specific, opinionated, occasionally understated, never promotional. (B13)

## Goal
For the micro-intent selected from Stage A's output, write a paragraph
simulating a first-hand "player hands-on" scenario built around that intent. (B10)

## Instructions
- **Constraints:**
  - Must include, at minimum, Sensory Details (what you saw/heard/felt using
    the platform) and Objective Critique (a genuinely two-sided assessment,
    not pure praise) (B11).
  - Must not produce "As an AI language model…"-style disclaimers or hollow,
    exaggerated praise — write with the specificity and mild skepticism of a
    real player, not marketing copy (B12).
- **Reference:**
  - Input: the micro-intent selected from Stage A's output (component C2),
    including its stated evidence/rationale.
- **Output shape** (feeds B16):
  - One paragraph (or short section) of first-hand-voiced content addressing
    the selected intent.
- **Self-check before finalizing** (from B20, B22):
  - Could this paragraph have been written about any {keyword} platform, or does
    it contain specific, hard-to-fabricate detail (B20)?
  - Re-read for generic AI filler ("overall, a great experience", "highly
    recommend") and cut it — that is exactly the failure mode B22 is testing
    for (Independent Thinking / catching the model's own logic flaws)."""

STAGE_C_AUDIT_TEMPLATE = """<!--
Traceability: prompts/master-prompt.md → build-topic-executables, {#}=1, component C4a
Source: topic-1-prompt-chaining/docs/problem-statement-categorized.md
  Goal: B15 (check half); together with C4b realizes B14
Type: dynamic

Run note: this previously stated "[娛樂城] content is YMYL-adjacent because
it involves the reader's money" as an assumed fact — true for 娛樂城
specifically, but not something that holds unconditionally for an arbitrary
substituted keyword (e.g. a keyword with no financial/health/safety/legal
dimension isn't YMYL-adjacent at all). Rewritten below as an instruction to
assess YMYL-relevance for the given keyword rather than assume it, so the
audit stays honest when {keyword} is generalized (src/prompts.py's
stage_c_audit_prompt).
-->

# Stage C — Compliance Audit

## Role
You are a senior SEO systems engineer: someone who combines deep, practical SEO
knowledge with the ability to design and build the AI systems that act on that
knowledge. You don't just know what makes content rank or what makes content
trustworthy — you understand that in this domain, those two forces are
frequently in tension, and you build systems whose job is to operate inside
that tension rather than pretend it doesn't exist.

(Default persona from the categorized doc's Role section — no audit-specific
persona was given in the source; the reviewer framing comes from the Goal and
Instructions below, not from a distinct persona.)

## Goal
Check whether Stage B's current paragraph complies with EEAT/YMYL principles,
and identify how to improve its Authoritativeness signal. (B15)

## Instructions
- **Constraints:**
  - This is the audit half of a Self-Correction chain (B14) — produce a
    structured verdict, not a rewritten paragraph (that is C4b's job).
  - Judge against EEAT (Experience, Expertise, Authoritativeness,
    Trustworthiness) and YMYL (Your Money or Your Life) principles; assess
    whether [{keyword}] content is YMYL-adjacent (e.g. because it involves
    the reader's money, health, safety, or legal standing) and weight the
    audit's rigor accordingly.
- **Reference:**
  - Input: Stage B's current paragraph — either the original, or a prior
    revision from C4b if this is a repeat pass in the orchestrator's loop.
- **Output shape:**
  - `{pass: bool, reasons: [...], authoritativeness_suggestions: [...]}`.
  - Each item in `reasons` must be specific enough for C4b (or a human) to act
    on — "reads too promotional" is not sufficient; name the sentence/claim
    and the principle it violates.
- **Self-check before finalizing** (from B21, B22):
  - Does every flagged reason trace to a concrete EEAT/YMYL principle rather
    than a vague stylistic preference (B21)?
  - Are you identifying real compliance/authority gaps through independent
    reasoning, not just restating that the text "could be improved" (B22)?"""

STAGE_C_CORRECTION_TEMPLATE = """<!--
Traceability: prompts/master-prompt.md → build-topic-executables, {#}=1, component C4b
Source: topic-1-prompt-chaining/docs/problem-statement-categorized.md
  Goal: B15 (correct half); together with C4a realizes B14
Type: dynamic
-->

# Stage C — Compliance Correction

## Role
You are a senior SEO systems engineer: someone who combines deep, practical SEO
knowledge with the ability to design and build the AI systems that act on that
knowledge. You don't just know what makes content rank or what makes content
trustworthy — you understand that in this domain, those two forces are
frequently in tension, and you build systems whose job is to operate inside
that tension rather than pretend it doesn't exist.

(Default persona from the categorized doc's Role section — no
correction-specific persona was given in the source.)

## Goal
Rewrite Stage B's paragraph to resolve the issues raised by the Compliance
Audit (C4a), while preserving the required first-hand voice from Stage B. (B15)

## Instructions
- **Constraints:**
  - Must resolve every item in the Audit's `reasons` list — do not rewrite
    unrelated parts of the paragraph.
  - Must NOT reintroduce what Stage B was built to avoid: AI disclaimers or
    hollow praise (B12) — a correction pass that fixes compliance but drifts
    back into generic marketing voice has failed both stages at once.
  - Must preserve the veteran-player persona and the required Sensory
    Details/Objective Critique from B11/B13 — this is a revision, not a
    rewrite from scratch.
- **Reference:**
  - Input: Stage B's paragraph plus the Audit's structured verdict from C4a.
- **Output shape:**
  - The revised paragraph, in the same form Stage B produced (feeds back into
    C4a for re-audit, per the orchestrator's loop — see component-specs.md).
- **Self-check before finalizing** (from B21):
  - Would this revision plausibly pass its own re-audit, or does it just
    relocate the same problem (e.g. trading one overclaim for another)?"""


def _render(template: str, keyword: str) -> str:
    # Plain string replacement (not str.format) — templates contain unrelated
    # literal {...} (e.g. the Output shape's {intent, evidence, ...}) that
    # str.format would try to interpret as format fields and fail on.
    return template.replace("{keyword}", keyword)


def stage_a_prompt(keyword: str) -> str:
    template = _render(STAGE_A_TEMPLATE, keyword)
    return (
        f"{template}\n\n---\nKeyword for this run: {keyword}\n\n"
        'Respond with ONLY a JSON object: {"intents": [{"intent": ..., '
        '"evidence": ..., "competitive_rationale": ...}, ...]} with at '
        "least 4 entries."
    )


def stage_b_prompt(selected_intent: dict, keyword: str) -> str:
    template = _render(STAGE_B_TEMPLATE, keyword)
    return (
        f"{template}\n\n---\nSelected micro-intent for this run:\n"
        f"{json.dumps(selected_intent, ensure_ascii=False, indent=2)}\n\n"
        "Respond with ONLY the paragraph text, no JSON wrapper."
    )


def stage_c_audit_prompt(paragraph: str, keyword: str) -> str:
    template = _render(STAGE_C_AUDIT_TEMPLATE, keyword)
    return (
        f"{template}\n\n---\nParagraph to audit:\n{paragraph}\n\n"
        'Respond with ONLY a JSON object: {"pass": bool, "reasons": [...], '
        '"authoritativeness_suggestions": [...]}.'
    )


def stage_c_correction_prompt(paragraph: str, reasons: list) -> str:
    reasons_block = "\n".join(f"- {r}" for r in reasons)
    return (
        f"{STAGE_C_CORRECTION_TEMPLATE}\n\n---\nParagraph to revise:\n{paragraph}\n\n"
        f"Audit reasons to resolve:\n{reasons_block}\n\n"
        "Respond with ONLY the revised paragraph text, no JSON wrapper."
    )


## 4. LLM client — Azure OpenAI backend

**Adapted** from `src/llm_client.py`'s `AzureOpenAILLMClient` — not a verbatim port like the cells above, because `__init__` reads config from `os.environ` (already populated by Cell 1 from Colab Secrets) rather than re-doing the .env-placeholder detection (`"your-"`/`"<"` checks) that only makes sense for a real `.env` file. **`complete()`'s body is meant to stay identical to the real class — if you change how the API is called there (e.g. the `max_completion_tokens` fix below), update both places.** Requests Azure/OpenAI JSON mode for the two components whose contract requires strict JSON (`stage_a` → `StageAOutput`, `stage_c_audit` → `AuditVerdict`) — the orchestrator's `_parse_json_response` below is a bare `json.loads` with no markdown-fence stripping, so this matters for reliability.

In [ ]:
class AzureOpenAILLMClient:
    _JSON_COMPONENTS = {"stage_a", "stage_c_audit"}

    def __init__(self) -> None:
        self._key = os.environ["AZURE_OPENAI_API_KEY"]
        self._endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
        self._api_version = os.environ["AZURE_OPENAI_API_VERSION"]
        self._deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

    def complete(self, prompt: str, *, component: str) -> str:
        from openai import AzureOpenAI

        client = AzureOpenAI(
            api_key=self._key,
            azure_endpoint=self._endpoint,
            api_version=self._api_version,
        )
        extra: dict = {}
        if component in self._JSON_COMPONENTS:
            extra["response_format"] = {"type": "json_object"}
        response = client.chat.completions.create(
            model=self._deployment,  # Azure takes the deployment name here, not a model name
            messages=[{"role": "user", "content": prompt}],
            # max_completion_tokens, not max_tokens — newer Azure/OpenAI chat
            # deployments (reasoning-tuned and GPT-5-class models included)
            # reject max_tokens outright and require this instead.
            max_completion_tokens=2048,
            **extra,
        )
        return response.choices[0].message.content


## 5. Deterministic orchestrator

Ported verbatim from `src/orchestrator.py`, including the `"type": "draft"` audit-trail entry that captures Stage B's pre-correction text (added so the original wording survives even when the audit/correct loop runs). Only `SESSIONS_DIR` is adapted, to a Colab-local `./sessions` folder.

In [ ]:
SESSIONS_DIR = Path("sessions")  # Colab-local; ephemeral — download before the runtime resets


@dataclass
class IntentSelectionRequest:
    session_id: str
    stage_a_intents: list


@dataclass
class ChainResult:
    stage_a_intents: list
    selected_intent: dict
    stage_b_final: str
    audit_trail: list = field(default_factory=list)
    final_pass: bool = False


def _parse_json_response(raw: str) -> Any:
    """Deterministic parsing gate: each dynamic component is instructed to
    return JSON; this is where that contract is enforced rather than
    trusted blindly (ties to B21 — logical consistency)."""
    try:
        return json.loads(raw)
    except json.JSONDecodeError as exc:
        raise ValueError(f"Component did not return valid JSON: {exc}\nRaw: {raw!r}") from exc


def _session_path(session_id: str) -> Path:
    return SESSIONS_DIR / f"{session_id}.json"


def _save_session(session_id: str, data: dict) -> None:
    SESSIONS_DIR.mkdir(parents=True, exist_ok=True)
    _session_path(session_id).write_text(
        json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8"
    )


def _load_session(session_id: str) -> dict:
    path = _session_path(session_id)
    if not path.exists():
        raise ValueError(f"Unknown session_id: {session_id}")
    return json.loads(path.read_text(encoding="utf-8"))


def start_prompt_chain(keyword: str, llm) -> IntentSelectionRequest:
    """Step 1 of C1 — runs Stage A (C2), pauses at the SEL human checkpoint."""
    raw = llm.complete(stage_a_prompt(keyword), component="stage_a")
    validated = StageAOutput.model_validate(_parse_json_response(raw))

    session_id = str(uuid.uuid4())
    intents = [i.model_dump() for i in validated.intents]
    _save_session(session_id, {"keyword": keyword, "stage_a_intents": intents})
    return IntentSelectionRequest(session_id=session_id, stage_a_intents=intents)


def resume_prompt_chain(
    session_id: str,
    selected_intent: dict,
    llm,
    retry_cap: int = 3,
) -> ChainResult:
    """Step 2 of C1 — resumes after the human's SEL choice, runs Stage B then
    the Stage C audit/correct loop (capped at retry_cap cycles)."""
    session = _load_session(session_id)
    if selected_intent not in session["stage_a_intents"]:
        raise ValueError("selected_intent is not one of this session's stage_a_intents")
    keyword = session["keyword"]

    raw_b = llm.complete(stage_b_prompt(selected_intent, keyword), component="stage_b")
    current_paragraph = StageBOutput(paragraph=raw_b.strip()).paragraph

    audit_trail: list = [{"type": "draft", "paragraph": current_paragraph}]
    verdict = AuditVerdict(**{"pass": False})
    for _ in range(retry_cap):
        raw_verdict = llm.complete(
            stage_c_audit_prompt(current_paragraph, keyword), component="stage_c_audit"
        )
        verdict = AuditVerdict.model_validate(_parse_json_response(raw_verdict))
        audit_trail.append({"type": "audit", **verdict.model_dump(by_alias=True)})
        if verdict.passed:
            break
        raw_correction = llm.complete(
            stage_c_correction_prompt(current_paragraph, verdict.reasons),
            component="stage_c_correction",
        )
        current_paragraph = StageCCorrection(paragraph=raw_correction.strip()).paragraph
        audit_trail.append({"type": "correction", "paragraph": current_paragraph})
    else:
        # for/else: runs only if the loop exhausted retry_cap without ever
        # hitting `break` (never passed). Without this, the last correction
        # above would be returned as stage_b_final never itself audited —
        # a real gap found via a live Azure run (see docs/live-run.md).
        raw_verdict = llm.complete(
            stage_c_audit_prompt(current_paragraph, keyword), component="stage_c_audit"
        )
        verdict = AuditVerdict.model_validate(_parse_json_response(raw_verdict))
        audit_trail.append({"type": "audit", **verdict.model_dump(by_alias=True)})

    result = ChainResult(
        stage_a_intents=session["stage_a_intents"],
        selected_intent=selected_intent,
        stage_b_final=current_paragraph,
        audit_trail=audit_trail,
        final_pass=verdict.passed,
    )
    _save_session(session_id, {**session, "result": asdict(result)})
    return result


## 6. Run Stage A

Change `KEYWORD` below to run against a different seed keyword. This makes one live call to your Azure deployment and prints the candidate micro-intents.

In [ ]:
KEYWORD = "娛樂城"  # change this to run against a different keyword

llm = AzureOpenAILLMClient()
req = start_prompt_chain(KEYWORD, llm)

print(f"Candidate micro-intents for [{KEYWORD}]:\n")
for i, intent in enumerate(req.stage_a_intents):
    print(f"[{i}] {intent['intent']}")
    print(f"    evidence: {intent['evidence'][:140]}...")
    print()


## 7. Human checkpoint (SEL) — pick one intent

This is the deliberate human-in-the-loop step from the source project's design (`component-specs.md`'s SEL checkpoint) — not automated, by design.

In [ ]:
choice = int(input("Pick one by index: "))
selected = req.stage_a_intents[choice]
print("Selected:", selected["intent"])


## 8. Run Stage B + the Stage C audit/correct loop

This makes one call for Stage B, then alternates audit/correction calls (capped at `retry_cap=3`) until the audit passes or the cap is hit.

In [ ]:
result = resume_prompt_chain(req.session_id, selected, llm, retry_cap=3)

print("=== Final paragraph (post-correction, if any) ===")
print(result.stage_b_final)
print()
print("Final pass:", result.final_pass)
print(f"Audit trail entries: {len(result.audit_trail)}")
print()
print("=== Audit trail ===")
for i, entry in enumerate(result.audit_trail):
    print(f"[{i}] {entry['type']}")
    if entry["type"] == "audit":
        print("    pass:", entry["pass"])
        for r in entry.get("reasons", []):
            print("    - reason:", r)
    elif entry["type"] in ("draft", "correction"):
        preview = entry["paragraph"][:120].replace("\n", " ")
        print(f"    paragraph: {preview}...")


## 9. Download this session

Colab's filesystem is ephemeral — this downloads the full session JSON (Stage A intents, the selected one, the pre-correction draft, every audit/correction cycle, and the final paragraph) to your machine before the runtime can recycle it.

In [ ]:
from google.colab import files

session_file = f"sessions/{req.session_id}.json"
print("Session saved at:", session_file)
files.download(session_file)
